In [1]:
import os
import re
import json
import fitz 
import asyncio
import pandas as pd
from glob import glob
from tqdm import tqdm
from pathlib import Path
from tqdm import tqdm
from tqdm.asyncio import tqdm_asyncio
from mistralai import Mistral
from dotenv import load_dotenv
from types import SimpleNamespace

from openai import OpenAI, AsyncOpenAI

load_dotenv()
api_key = os.environ["MISTRAL_API_KEY"]

client = Mistral(api_key=api_key)
gpt_client = OpenAI()

In [23]:
def process_filename(file_path: str | Path, limit_folder=None) -> str:
    path_obj = Path(file_path).resolve()
    return f"\\\\?\\{path_obj}"

def upload_pdf_mistral(filepath: str):
    safe_path = process_filename(filepath)
    
    with open(safe_path, "rb") as f:
        uploaded_pdf = client.files.upload(
            file={
                "file_name": Path(filepath).name,
                "content": f,
            },
            purpose="ocr"
        )
    
    signed_url = client.files.get_signed_url(file_id=uploaded_pdf.id)
    return signed_url.url

def parse_pdf_text(filepath: str):
    safe_path = process_filename(filepath)
    
    try:
        with open(safe_path, "rb") as f:
            file_bytes = f.read()
            
        doc = fitz.open(stream=file_bytes, filetype="pdf")
        
        pages_list = []
        for page in doc:
            text_content = page.get_text("text") 
            
            page_obj = SimpleNamespace(markdown=text_content)
            pages_list.append(page_obj)
        
        return SimpleNamespace(pages=pages_list)
        
    except OSError as e:
        print(f"Error opening {Path(filepath).name}: {e}")
        return None

def parse_pdf_ocr(filepath: str):
    try:
        ocr_response = client.ocr.process(
            model="mistral-ocr-latest",
            document={
                "type": "document_url",
                "document_url": upload_pdf_mistral(filepath=filepath),
            },
            include_image_base64=True,
        )
        return ocr_response
    except Exception as e:
        print(f"Error OCR-ing {filepath}: {e}")
        return None

def parse_pdf(filepath: str, method: str = "text"):
    if method == "ocr":
        return parse_pdf_ocr(filepath)
    elif method == "text":
        return parse_pdf_text(filepath)
    else:
        raise ValueError("Method must be 'ocr' or 'text'")


In [24]:
sample_judgement_path = r"C:\Users\ghana\OneDrive\kuliah\Semester 8\TA\experiment\downloads\Putusan_PN_DENPASAR_Nomor_1259_Pdt.G_2024_PN_Dps.pdf"
responses = parse_pdf(sample_judgement_path, method="text")
print(responses.pages[-1].markdown)

hkama
ahkamah Agung Repub
ahkamah Agung Republik Indonesia
mah Agung Republik Indonesia
blik Indonesi
Direktori Putusan Mahkamah Agung Republik Indonesia
putusan.mahkamahagung.go.id
 I Ketut Suarta, S.H. 
Panitera Pengganti,
TTD
           Ni Wayan Meidayanti S.H.
  Perincian  biaya:
1.   Biaya Pendaftaran ………………..  Rp.   30.000,00  
2.   Biaya pemberkasan.………………  Rp. 100.000,00
3.   Biaya Penggandaan ……………...  Rp.   40.000,00
3.   Biaya panggilan  ..........................
Rp.   64.000,00
4.   PNBP ……………………………….  Rp   20.000,00
5.   Redaksi putusan  …………………. Rp.  10.000,00
6.   Meterai……………………………...    
 
 Rp.   
 
 10.000,00
 
 
      Jumlah ……………………………… Rp. 274.000,00 
                  (Dua ratus tujuh puluh empat ribu rupiah);  
Halaman 13 dari 13 Putusan Nomor: 1259/Pdt.G/2024/PN Dps
Disclaimer
Kepaniteraan Mahkamah Agung Republik Indonesia berusaha untuk selalu mencantumkan informasi paling kini dan akurat sebagai bentuk komitmen Mahkamah Agung untuk pelayanan publik, transpara

In [25]:
judgement_summarization_prompt = """
You are a legal data assistant specializing in Indonesian Court Decisions. 
Process the provided judgment text and output a JSON object containing exactly two fields:

1. "incidents": 
Extract legal incidents from a given judgment to understand why specific sections or articles of law were
cited. These extracted incidents will later be matched with relevant sections and articles.
Please process the given legal judgment and focus on the following instructions:
Objective: Identify and extract all legal incidents referenced in the judgment, focusing on the key facts
and legal issues of the case.
Structure: Phrase each incident concisely and neutrally. Exclude case-specific details (e.g., names, dates,
case numbers). The extracted incidents should be rich in legal reasoning and sufficiently descriptive to
enable accurate section/article matching.
Focus Areas: Capture the core facts and issues underlying the case.
Language: Bahasa Indonesia.

2. "relevant_laws":
Objective: Extract the specific Articles (Pasal) and Regulations (UU, KUHP, etc.) used by the judge as the legal basis for the decision.
Format: A list of strings. Each string should contain the specific article and regulation (e.g., "Pasal 1365 KUHPerdata").
Constraint: Only include laws explicitly cited in the judge's consideration or decision part. Only include regulation that have article number being referred to.
Output: [REGULATION_NAME] Pasal [ARTICLE_NUMBER] -> [KUHAPerdata] Pasal 100

Output must be valid JSON only. Do not add markdown formatting (```json).
"""

In [27]:
gpt_client = AsyncOpenAI()
LLM_MODEL = "gpt-5-mini"
BATCH_SIZE = 10
sem = asyncio.Semaphore(BATCH_SIZE)
output_path = r"C:\Users\ghana\OneDrive\kuliah\Semester 8\TA\data\judgement\judgement_to_content.json"

async def process_single_judgement(filepath):
    async with sem:  
        try:
            responses = await asyncio.to_thread(parse_pdf, filepath)
            
            if not responses:
                return filepath, None
            
            full_judgment_text = "\n\n".join([page.markdown for page in responses.pages])
            if "kuhp" not in full_judgment_text.lower():
                return filepath, None

            completion = await gpt_client.chat.completions.create(
                model=LLM_MODEL, 
                messages=[
                    {
                        "role": "user", 
                        "content": f"{judgement_summarization_prompt}\n\nJudgement Text:\n\n{full_judgment_text}"
                    }
                ],
                response_format={"type": "json_object"}
            )
            raw_content = completion.choices[0].message.content
            
            try:
                content = json.loads(raw_content)
            except json.JSONDecodeError:
                print(f"Got invalid json on {filepath}, calling llm to fix")
                fix_completion = await gpt_client.chat.completions.create(
                    model=LLM_MODEL, 
                    messages=[
                        {
                            "role": "user", 
                            "content": f"Fix the malformed json below into valid json:\n\n{raw_content}"
                        }
                    ],
                    response_format={"type": "json_object"}
                )
                content = json.loads(fix_completion.choices[0].message.content)
            
            return filepath, content

        except Exception as e:
            print(f"Error processing {filepath}: {e}")
            return filepath, None

judgement_filepaths = glob("downloads/*.pdf")
judgement_to_content = {}

try:
    with open(output_path, "r") as f:
        judgement_to_content = json.load(f)
except FileNotFoundError:
    pass

files_to_process = [f for f in judgement_filepaths if f not in judgement_to_content]
tasks = [process_single_judgement(f) for f in files_to_process]

if tasks:
    for future in tqdm_asyncio.as_completed(tasks, total=len(tasks)):
        filepath, content = await future
        if content:
            judgement_to_content[filepath] = content

    with open(output_path, "w") as f:
        json.dump(judgement_to_content, f, indent=2)
else:
    print("No new files to process.")

 11%|█         | 393/3721 [02:46<10:13,  5.42it/s]  

Error processing downloads\Putusan_PN_DENPASAR_Nomor_294_Pdt.G_2024_PN_Dps.pdf: Error code: 400 - {'error': {'message': 'Input tokens exceed the configured limit of 272000 tokens. Your messages resulted in 288262 tokens. Please reduce the length of the messages.', 'type': 'invalid_request_error', 'param': 'messages', 'code': 'context_length_exceeded'}}


 25%|██▍       | 918/3721 [05:06<06:51,  6.82it/s]  

Error processing downloads\Putusan_PN_DENPASAR_Nomor_205_Pdt.G_2024_PN_Dps.pdf: Error code: 400 - {'error': {'message': 'Input tokens exceed the configured limit of 272000 tokens. Your messages resulted in 426575 tokens. Please reduce the length of the messages.', 'type': 'invalid_request_error', 'param': 'messages', 'code': 'context_length_exceeded'}}


100%|██████████| 3721/3721 [25:53<00:00,  2.39it/s]  


In [18]:
# with open(r"C:\Users\ghana\OneDrive\kuliah\Semester 8\TA\data\judgement\judgement_to_content.json", "w") as f:
#     json.dump(judgement_to_content, f)

# read judgement json
import json
with open(r"C:\Users\ghana\OneDrive\kuliah\Semester 8\TA\data\judgement\judgement_to_content.json", "r") as f:
    judgement_to_content = json.load(f)

for i in range(500):
    k = list(judgement_to_content.keys())[i]
    content = judgement_to_content[k]
    if "perdata" in (" ".join(content["relevant_laws"])).lower():
        print("\n".join(content["incidents"]))

        print()
        print(content["relevant_laws"])
        break

Sengketa kepemilikan dan penguasaan dua unit alat berat yang dipersengketakan antara pihak yang mengklaim hak milik berdasarkan perjanjian pengelolaan kebun dan pihak lain yang masih menguasai alat setelah berakhirnya perjanjian.
Tuntutan restitusi (penyerahan kembali) barang bergerak milik penggugat dan tuntutan ganti rugi atas penguasaan tanpa hak serta penggunaan alat berat oleh pihak yang menguasai.
Permintaan pengakuan dan pelaksanaan sita revindicatoir terhadap objek sengketa sebagai upaya mengamankan hak milik atas barang bergerak.
Keabsahan surat kuasa yang digunakan untuk menarik/mengumpulkan peralatan dan inventaris dipersengketakan; pengadilan menilai surat kuasa tersebut tidak sah dan bertentangan dengan hukum.
Pihak tergugat mengajukan eksepsi yaitu keberatan prosedural/ne bis in idem terhadap gugatan pokok penggugat.
Tuntutan rekonvensi oleh pihak penguasa alat yang menuntut ganti rugi materiil dan imateriil atas kerusakan tempat serta biaya sewa, jasa pengamanan, perbaik

In [ ]:
conten

{'incidents': ['Perjanjian kredit multiple antara debitur dan bank yang dijamin dengan hak tanggungan atas beberapa bidang tanah dan bangunan, serta adanya akta-akta perjanjian kredit dan akta pembebanan hak tanggungan yang diterbitkan oleh notaris/PPAT.',
  'Klaim bahwa salinan akta notaris/PPAT dan dokumen administrasi kredit tidak pernah diserahkan kepada debitur/penjamin oleh kreditur, sehingga menghambat kemampuan debitur/penjamin untuk mengalihkan atau menebus jaminan ke pihak ketiga.',
  'Sengketa mengenai keberadaan, keaslian dan/atau tanggal suatu akta perjanjian kredit awal yang diduga ditandatangani di hadapan notaris lain, ditolak atau tidak diakui oleh pihak penggugat.',
  'Bank mengirimkan surat peringatan dan menagih tunggakan serta menghitung bunga dan denda; penggugat mempertanyakan dasar hukum dan perhitungan tersebut dengan dalih perjanjian mulai berlaku pada tanggal berbeda.',
  'Wanprestasi debitur/penjamin atas perjanjian kredit yang mengakibatkan kreditur mengamb

: 